# Demand Forecasting Pipeline — XGBoost with Business Features

**Goal:** Improve on the Prophet baseline by treating demand forecasting as a supervised learning problem, adding business-relevant features (discount rate, seasonality).

**Key difference from v1:**  
Prophet learns time patterns automatically. XGBoost requires us to engineer those patterns as explicit features — lag values, rolling averages, calendar signals. This gives us more control, and lets us plug in business signals like promotions.

**Pipeline steps:**
1. Load & aggregate (same as v1, but also keep discount rate)
2. Feature engineering: lags, rolling windows, calendar, discount
3. Train/test split
4. Train XGBoost, evaluate vs Prophet baseline
5. Feature importance — which signals actually drive demand?
6. Reorder decision logic (same structure as v1)

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

DATA_PATH        = 'DataCoSupplyChainDataset.csv'
TEST_WEEKS       = 8
FORECAST_HORIZON = 8
LEAD_TIME_WEEKS  = 2
SERVICE_LEVEL    = 1.65

## 1. Load & Aggregate

Same as v1, with one addition: we keep `Order Item Discount Rate`  
and aggregate it as the **average discount applied that week per SKU**.

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding='latin1',
                  usecols=['order date (DateOrders)',
                           'Product Card Id',
                           'Product Name',
                           'Order Item Quantity',
                           'Order Item Discount Rate'])

raw['order_date'] = pd.to_datetime(raw['order date (DateOrders)'])
raw['week']       = raw['order_date'].dt.to_period('W').dt.start_time

weekly = (
    raw
    .groupby(['Product Card Id', 'Product Name', 'week'])
    .agg(
        units_sold   = ('Order Item Quantity',    'sum'),
        avg_discount = ('Order Item Discount Rate','mean')
    )
    .reset_index()
    .rename(columns={'Product Card Id': 'product_id',
                     'Product Name':    'product_name'})
    .sort_values(['product_id', 'week'])
    .reset_index(drop=True)
)

# Drop last incomplete week (same reason as v1)
weekly = weekly[weekly['week'] < weekly['week'].max()].copy()

# Keep SKUs with full history
week_counts = weekly.groupby('product_id')['week'].count()
full_skus   = week_counts[week_counts >= 130].index
weekly      = weekly[weekly['product_id'].isin(full_skus)].copy()

sku_lookup = (weekly[['product_id','product_name']]
              .drop_duplicates()
              .set_index('product_id')['product_name'])

print(f"Rows: {len(weekly):,} | SKUs: {weekly['product_id'].nunique()}")
print(f"Date range: {weekly['week'].min().date()} → {weekly['week'].max().date()}")

## 2. Feature Engineering

XGBoost cannot learn from raw timestamps — we need to encode time patterns as numeric features.

| Feature | Type | What it captures |
|---|---|---|
| `lag_1` … `lag_4` | Autoregressive | Last 4 weeks of actual demand |
| `rolling_mean_4` | Trend | Short-term demand level |
| `rolling_mean_12` | Trend | Long-term demand level |
| `rolling_std_4` | Volatility | Recent demand variability |
| `month`, `quarter` | Calendar | Seasonal patterns |
| `avg_discount` | Business signal | Promotion effect on demand |

> **Why shift(1)?** Rolling calculations use `shift(1)` to ensure we never leak future information into the features — at prediction time, we only know what happened *before* the current week.

In [ ]:
FEATURE_COLS = ['lag_1','lag_2','lag_3','lag_4',
                'rolling_mean_4','rolling_mean_12','rolling_std_4',
                'month','quarter','avg_discount']

def add_features(df_sku):
    """Add lag and calendar features to a single SKU's time series."""
    df = df_sku.copy().sort_values('week').reset_index(drop=True)

    # Autoregressive lags
    for lag in [1, 2, 3, 4]:
        df[f'lag_{lag}'] = df['units_sold'].shift(lag)

    # Rolling statistics (shift(1) to avoid data leakage)
    shifted = df['units_sold'].shift(1)
    df['rolling_mean_4']  = shifted.rolling(4).mean()
    df['rolling_mean_12'] = shifted.rolling(12).mean()
    df['rolling_std_4']   = shifted.rolling(4).std()

    # Calendar features
    df['month']   = df['week'].dt.month
    df['quarter'] = df['week'].dt.quarter

    # Drop rows where lags are undefined
    df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    return df


# Apply to all SKUs
all_featured = pd.concat(
    [add_features(weekly[weekly['product_id'] == pid]) for pid in full_skus],
    ignore_index=True
)

print(f"Rows after feature engineering: {len(all_featured):,}")
print(f"Features: {FEATURE_COLS}")
print()
print(all_featured[['week','units_sold'] + FEATURE_COLS].head(3).to_string())

## 3. Train / Test Split

We split **per SKU** — the last 8 weeks of each SKU form its test set.  
The model trains on all SKUs combined: it learns shared demand patterns across products.

In [ ]:
def split_sku(df_sku, test_weeks=TEST_WEEKS):
    df_sku = df_sku.sort_values('week').reset_index(drop=True)
    return df_sku.iloc[:-test_weeks].copy(), df_sku.iloc[-test_weeks:].copy()

train_parts, test_parts = [], []
for pid in full_skus:
    tr, te = split_sku(all_featured[all_featured['product_id'] == pid])
    train_parts.append(tr)
    test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts,  ignore_index=True)

X_train = train_df[FEATURE_COLS]
y_train = train_df['units_sold']
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['units_sold']

print(f"Train: {len(X_train)} rows ({len(full_skus)} SKUs × ~{len(X_train)//len(full_skus)} weeks)")
print(f"Test : {len(X_test)} rows ({len(full_skus)} SKUs × {TEST_WEEKS} weeks)")

## 4. Train XGBoost & Evaluate

We use a global model: trained on all SKUs at once.  
This is a deliberate design choice — shared patterns (seasonality, discount response) are learned across all products, which also helps with cold-start situations where a SKU has limited history.

In [ ]:
model = XGBRegressor(
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 4,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    random_state      = 42,
    verbosity         = 0
)

model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)],
          verbose=False)

test_df = test_df.copy()
test_df['yhat'] = model.predict(X_test).clip(min=0)

print("Model trained.")

In [ ]:
# Evaluate per SKU + compare against Prophet baseline
prophet_mape = {
    191: 165.6, 365: 177.3, 403: 230.0, 502: 229.8,
    627: 518.1, 957: 302.4, 1004: 480.2, 1014: 229.9, 1073: 166.1
}

eval_rows = []
for pid in full_skus:
    sku_test = test_df[test_df['product_id'] == pid]
    actual   = sku_test['units_sold'].values
    pred     = sku_test['yhat'].values

    mae  = mean_absolute_error(actual, pred)
    nz   = actual != 0
    mape = np.mean(np.abs((actual[nz] - pred[nz]) / actual[nz])) * 100

    eval_rows.append({
        'product_id':    pid,
        'product_name':  sku_lookup[pid][:40],
        'MAE_xgb':       round(mae, 1),
        'MAPE_xgb_%':    round(mape, 1),
        'MAPE_prophet_%':prophet_mape[pid],
    })

eval_df = pd.DataFrame(eval_rows)
eval_df['delta_%'] = (eval_df['MAPE_xgb_%'] - eval_df['MAPE_prophet_%']).round(1)

print(eval_df.to_string(index=False))
print()
print(f"Median MAE   XGBoost : {eval_df['MAE_xgb'].median():.1f} units/week")
print(f"Median MAPE  XGBoost : {eval_df['MAPE_xgb_%'].median():.1f}%")
print(f"Median MAPE  Prophet : {eval_df['MAPE_prophet_%'].median():.1f}%")
print()
better = (eval_df['delta_%'] < 0).sum()
print(f"XGBoost better than Prophet on {better}/{len(eval_df)} SKUs")

In [ ]:
# Visual comparison: MAPE side by side
fig, ax = plt.subplots(figsize=(11, 4))
x     = np.arange(len(eval_df))
width = 0.35
names = [n[:22] for n in eval_df['product_name']]

bars1 = ax.bar(x - width/2, eval_df['MAPE_prophet_%'], width,
               label='Prophet', color='#93c5fd', alpha=0.85)
bars2 = ax.bar(x + width/2, eval_df['MAPE_xgb_%'], width,
               label='XGBoost', color='#2563eb', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('MAPE (%)')
ax.set_title('MAPE comparison: Prophet vs XGBoost (8-week test set)', fontsize=12)
ax.legend()
ax.axhline(100, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
plt.tight_layout()
plt.savefig('mape_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Forecast vs actual for best and worst SKU
best_pid  = eval_df.loc[eval_df['MAPE_xgb_%'].idxmin(), 'product_id']
worst_pid = eval_df.loc[eval_df['MAPE_xgb_%'].idxmax(), 'product_id']

def plot_sku(pid, ax, label=''):
    sku_all   = all_featured[all_featured['product_id'] == pid].sort_values('week')
    sku_train = train_df[train_df['product_id'] == pid].sort_values('week')
    sku_test  = test_df[test_df['product_id'] == pid].sort_values('week')

    ax.plot(sku_train['week'], sku_train['units_sold'],
            color='#374151', linewidth=1, label='Actual (train)')
    ax.plot(sku_test['week'], sku_test['units_sold'],
            color='#ef4444', linewidth=2, label='Actual (test)')
    ax.plot(sku_test['week'], sku_test['yhat'],
            color='#2563eb', linewidth=2, linestyle='--', label='XGBoost forecast')
    ax.axvline(sku_test['week'].min(), color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f"{sku_lookup[pid][:45]}  {label}", fontsize=11)
    ax.set_ylabel('Units sold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=9)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
plot_sku(best_pid,  axes[0], '[best MAPE]')
plot_sku(worst_pid, axes[1], '[worst MAPE]')
plt.tight_layout()
plt.savefig('xgb_forecast_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance

Which signals actually drive demand predictions?  

This is directly relevant to stakeholder conversations:  
> *"Our model relies most heavily on recent sales history (lag features), but discount rate is the strongest external signal — meaning promotional planning has a measurable effect on demand."*

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ef4444' if 'discount' in f else
          '#f59e0b' if f in ('month','quarter') else
          '#2563eb'
          for f in importance.index]

importance.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Feature importance (gain)')
ax.set_title('What drives the XGBoost demand forecast?', fontsize=12)

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2563eb', label='Lag / rolling (autoregressive)'),
    Patch(facecolor='#f59e0b', label='Calendar (seasonality)'),
    Patch(facecolor='#ef4444', label='Discount (business signal)'),
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(importance.sort_values(ascending=False).round(4))

## 6. Reorder Decision Logic

Same structure as v1 — forecast feeds directly into a purchase suggestion script.

One difference: XGBoost gives us a **point estimate only** (no prediction interval).  
We estimate demand uncertainty from the model's residuals on the training set,  
and use that to calculate safety stock.

In [ ]:
def get_residual_std(pid):
    """Estimate forecast uncertainty from training residuals for one SKU."""
    sku_train = train_df[train_df['product_id'] == pid]
    preds     = model.predict(sku_train[FEATURE_COLS]).clip(min=0)
    residuals = sku_train['units_sold'].values - preds
    return residuals.std()


def reorder_decision_xgb(pid, current_stock, in_transit=0,
                          lead_time_weeks=LEAD_TIME_WEEKS,
                          z=SERVICE_LEVEL):
    """
    Generate a reorder recommendation using XGBoost predictions.
    Uses the last known feature row to forecast the next period.
    """
    # Use test-set predictions as our "current" forecast
    sku_test      = test_df[test_df['product_id'] == pid].sort_values('week')
    forecast_vals = sku_test['yhat'].values  # 8 weeks of predictions

    demand_lt     = forecast_vals[:lead_time_weeks].sum()
    residual_std  = get_residual_std(pid)
    safety_stock  = z * residual_std * np.sqrt(lead_time_weeks)
    reorder_point = demand_lt + safety_stock

    total_demand  = forecast_vals.sum()
    order_qty     = max(0, round(total_demand - current_stock - in_transit))
    should_reorder = (current_stock + in_transit) < reorder_point

    return {
        'product_name':    sku_lookup[pid],
        'current_stock':   current_stock,
        'reorder_point':   round(reorder_point),
        'safety_stock':    round(safety_stock),
        'forecast_8w':     round(total_demand),
        'suggested_order': order_qty,
        'action':          'REORDER' if should_reorder else 'ok',
    }

In [ ]:
# Simulate inventory levels (same seed as v1 for fair comparison)
rng = np.random.default_rng(seed=42)

suggestions = []
for pid in full_skus:
    avg_weekly  = train_df[train_df['product_id'] == pid]['units_sold'].mean()
    sim_stock   = int(avg_weekly * rng.uniform(1.5, 4.0))
    sim_transit = int(avg_weekly * rng.uniform(0, 1.0))

    rec = reorder_decision_xgb(pid, sim_stock, sim_transit)
    suggestions.append(rec)

suggestions_df = pd.DataFrame(suggestions).sort_values('action', ascending=False)
print(suggestions_df.to_string(index=False))

In [ ]:
# Final purchase order output
purchase_orders = suggestions_df[suggestions_df['action'] == 'REORDER']

print("=" * 60)
print(f"  AUTOMATED REORDER SUGGESTIONS — {pd.Timestamp.today().date()}")
print("=" * 60)

if len(purchase_orders) == 0:
    print("  All SKUs sufficiently stocked. No reorders needed.")
else:
    for _, row in purchase_orders.iterrows():
        print(f"  {row['product_name'][:45]:<45}"
              f"  order: {row['suggested_order']:>5} units"
              f"  (stock: {row['current_stock']}, ROP: {row['reorder_point']})")

print("=" * 60)
print(f"  {len(purchase_orders)} of {len(suggestions_df)} SKUs flagged for reorder")
print("=" * 60)

## Summary & Reflection

### What this notebook demonstrates
- Converting a time series problem into supervised learning using lag features
- Training a **global model** across all SKUs — shares patterns, handles cold-start better than per-SKU models
- Using business signals (discount rate) alongside historical patterns
- Connecting model output to an operational reorder decision

### Why the MAPE is still high
The test period ends with an abrupt demand drop in the last week of the dataset — a data artefact, not a real demand change. Both models are penalised for not predicting this. In a production system this would be detected as an anomaly and excluded from evaluation.

### Key finding: feature importance
Lag features dominate, as expected — recent sales are the strongest signal. `avg_discount` ranks above calendar features, confirming that promotional activity has a measurable effect on weekly demand. This supports the business case for including promotional calendars in future model versions.

### Next steps
- Add more SKUs by relaxing the history threshold and using cross-SKU features
- Include supplier lead time variability in safety stock calculation
- Add a monitoring layer: alert when actual demand deviates from forecast by >2σ